# Editing Simulation Parameters

FASTSim simulations are controlled by two sets of parameters:

1. **`SimParams`** - solver-level parameters that apply to all vehicle types, governing achieved speed and trace miss handling.
2. **`HEVSimulationParams`** - additional simulation parameters for HEV/PHEV vehicles.

## SimParams: Achieved Speed and Trace Miss Parameters

The `SimParams` class defines the behavior of the solver for achieved vehicle speed as well as parameters related to trace miss.

### Key Parameters

**Achieved Speed Solver:**

- **`ach_speed_max_iter`** : Maximum number of iterations for the achieved speed solver. Higher values allow more iterations to find a speed the vehicle can achieve, but increase computation time. Default: 3.
- **`ach_speed_tol`** : Tolerance on speed change between iterations (dimensionless ratio). Smaller values mean tighter tolerance and more accuracy, but longer convergence. Default: 0.001.
- **`ach_speed_solver_gain`** : Gain factor (0.0–1.0) for the Newton method speed update. Higher values (e.g., 0.9) lead to faster convergence but risk overshooting. Default: 0.9.

**Trace Miss Handling:**

For detailed examples and explanations, see [Handling Trace Miss](trace-miss.ipynb).

- **`trace_miss_opts`** : How to handle situations where the vehicle cannot achieve the target speed (`"Error"`, `"Allow"`, `"AllowChecked"`, `"Correct"`). Default: `"Error"`.
- **`trace_miss_tol`** : Tolerance parameters (distance and speed) that apply when `trace_miss_opts` is `"AllowChecked"`.
- **`trace_miss_correct_max_steps`** : When `trace_miss_opts` is `"Correct"`, the maximum number of steps allowed to re-synchronize with the trace. Must be ≥ 2. Default: 6.

**Other:**

- **`f2_const_air_density`** : Whether to use a constant air density (FASTSim-2 style) instead of computing it from elevation. Default: true.
- **`ambient_thermal_soak`** : If true, simulate an engine-off 'soak' for thermally-aware models. Default: false.

### Example:

In [22]:
import fastsim

# Create a SimParams object from default values
params = fastsim.SimParams.default().to_pydict()

print("Simulation params:")
params

Simulation params:


{'ach_speed_max_iter': 3,
 'ach_speed_tol': 0.001,
 'ach_speed_solver_gain': 0.9,
 'trace_miss_tol': {'tol_dist': 100.0,
  'tol_dist_frac': 0.05,
  'tol_speed': 10.0,
  'tol_speed_frac': 0.5},
 'trace_miss_opts': 'Error',
 'trace_miss_correct_max_steps': 6,
 'f2_const_air_density': True,
 'ambient_thermal_soak': False}

In [23]:
# Modify trace miss handling to allow some miss with tolerance
params['trace_miss_opts'] = 'AllowChecked'

# Load a vehicle and cycle
veh = fastsim.Vehicle.from_resource("2012_Ford_Fusion.yaml")
cyc = fastsim.Cycle.from_resource("udds.csv")

# Run simulation with modified parameters
sim_params = fastsim.SimParams.from_pydict(params)
sd = fastsim.SimDrive(veh, cyc, sim_params)
sd.walk()

print(f"Modified trace_miss_opts: {params['trace_miss_opts']}")
print("Simulation completed")

Modified trace_miss_opts: AllowChecked
Simulation completed


---

## HEV and PHEV Simulation Parameters

Hybrid Electric Vehicles (HEVs) and Plug-in Hybrid Electric Vehicles (PHEVs) have additional simulation parameters that control the hybrid powertrain strategy. These parameters are stored in the vehicle object and control how the hybrid system balances energy between the engine and battery.

**Parameters:**

- **`res_per_fuel_lim`** : Ratio of reversible energy storage (battery) energy capacity to fuel converter power output capability. Default: 0.005.
- **`balance_soc`** : If true, the solver will iterate to achieve a balanced state-of-charge at the end of the cycle (SOC approximately equals the initial SOC). If false, the simulation runs as-is without balancing. Default: true.
- **`soc_balance_iter_err`** : When `balance_soc` is true, the maximum number of iterations allowed to achieve SOC balance. If the solver cannot balance SOC within this many iterations, an error is raised. Default: 5.
- **`save_soc_bal_iters`** : Whether to save the history for each intermediate SOC balancing iteration (for debugging). Default: false.

## Example:

In [24]:
import fastsim

# Load an HEV vehicle
veh_hev = fastsim.Vehicle.from_resource("2016_TOYOTA_Prius_Two.yaml")

# Access HEV parameters via to_pydict()
veh_dict = veh_hev.to_pydict()
hev_params = veh_dict['pt_type']['HEV']['sim_params']

print("HEV simulation params:")
hev_params

HEV simulation params:


{'res_per_fuel_lim': 0.005,
 'soc_balance_iter_err': 5,
 'balance_soc': True,
 'save_soc_bal_iters': False}

In [25]:
# Modify SOC balancing to allow more iterations
veh_dict = veh_hev.to_pydict()
veh_dict['pt_type']['HEV']['sim_params']['soc_balance_iter_err'] = 10

# Recreate the vehicle with modified parameters
veh_hev_modified = fastsim.Vehicle.from_pydict(veh_dict)

# Run a simulation with the modified HEV parameters
cyc = fastsim.Cycle.from_resource("udds.csv")
sd = fastsim.SimDrive(veh_hev_modified, cyc)
sd.walk()

print(f"Modified soc_balance_iter_err: {veh_dict['pt_type']['HEV']['sim_params']['soc_balance_iter_err']}")
print("Simulation completed")

Modified soc_balance_iter_err: 10
Simulation completed
